# 🏠 California Housing Price Predictor
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/california-housing/training/notebook.ipynb)

Predict the **median house value** of a California census district (1990 census, 20,640 districts) from
8 features such as median income, house age and location. Four regressors are compared: LinearRegression,
RandomForest, **XGBoost** and a **PyTorch MLP**. The default model is picked by validation RMSE, and the test
split is scored once. XGBoost and the MLP are exported to the Hugging Face model folder that the
[Space](https://huggingface.co/spaces/shalev396/california-housing) serves.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "california-housing"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# 1d. Project code: every step below calls src/ (training) and ../model/model.py (inference, shared with the Space)
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

from src import data_setup, engine, export, model_builder, utils   # importing src puts ../model on sys.path
from src.config import Config
import model as M                                                   # ../model/model.py

print(utils.lib_versions("xgboost", "torch", "sklearn", "numpy", "pandas", "huggingface_hub"))

## 2. Config
Every hyperparameter lives in `src/config.py`. `SMOKE_TEST=1` shrinks the run to 3,000 rows, tiny models and
3 MLP epochs (about a minute on a CPU), and writes only to `outputs/smoke/`. `SMOKE_TEST=0` is the real run and
**overwrites `../model`**.

In [ ]:
import os

os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick smoke run; a SMOKE_TEST set outside the notebook wins
PUSH_TO_HUB = False                         # upload the exported model folder to the Hub (section 8)

cfg = Config()
utils.set_seeds(cfg.seed, framework="torch")
torch_device = torch.device(device)
print("smoke:", cfg.smoke, "| device:", torch_device, "| export to:", cfg.model_dir)
cfg

## 3. Data
`sklearn.datasets.fetch_california_housing` is downloaded once and cached as `training/data/california_housing.csv`
(fallback: the raw `housing.csv` of *Hands-On Machine Learning*, converted to the same features). The target
`MedHouseVal` is in units of $100k and is capped at 5.0 in the source data. `model.engineer` adds 3 ratios
(`rooms_per_household`, `bedrooms_per_room`, `population_per_household`), the same function the Space runs.
Splits: 80/20 train/test with seed 42, then 15 % of the train part as validation (early stopping + model selection).

In [ ]:
frame = data_setup.load_frame(cfg)
splits = data_setup.split(frame, cfg)
(X_tr, y_tr), (X_val, y_val), (X_test, y_test) = (data_setup.xy(splits[k]) for k in ("train", "val", "test"))
print("X_train", X_tr.shape, "| target ($100k): mean", round(y_tr.mean(), 3), "max", y_tr.max())
data_setup.with_features(frame).describe().T.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
sc = axes[0].scatter(frame["Longitude"], frame["Latitude"], c=frame[M.TARGET], s=3, cmap="viridis")
fig.colorbar(sc, ax=axes[0], label="MedHouseVal ($100k)")
axes[0].set(title="Price by location", xlabel="Longitude", ylabel="Latitude")
axes[1].hist(frame[M.TARGET], bins=60, color="#2a78d6")
axes[1].set(title="Target distribution (note the cap at 5.0)", xlabel="MedHouseVal ($100k)")
corr = data_setup.with_features(frame).corr()[M.TARGET].drop(M.TARGET).sort_values()
axes[2].barh(corr.index, corr.values, color=np.where(corr > 0, "#2a78d6", "#eb6834"))
axes[2].set(title="Correlation with MedHouseVal")
plt.tight_layout()
plt.show()

## 4. Load model
`model_builder.build_all` creates the four experiments. The MLP class `HousingMLP` comes from `../model/model.py`
(a `PyTorchModelHubMixin` module, so it saves as `model.safetensors` + `config.json`). The MLP's inputs are
standardized with a `StandardScaler` fit on the train split only; the tree models use raw features.

In [ ]:
models = model_builder.build_all(cfg)
scaler = model_builder.build_scaler().fit(X_tr)
for name in models:
    print(f"{name:18s} {model_builder.describe(name, cfg)}")
print(models["mlp"])
print(f"MLP parameters: {utils.count_params(models['mlp']):,}")

## 5. Training
Each experiment is fit on the train split. XGBoost stops after 50 rounds without validation improvement and is
trimmed to its best round; the MLP (AdamW, MSE) stops after 10 epochs without improvement and restores its best
weights. XGBoost and the MLP are the slow parts; the RandomForest takes seconds. The full run of 2026-09-25 took
about 37 min on a shared CPU with other jobs running (XGBoost 1,205 s, MLP 956 s, RandomForest 8 s; see
`README.md`), and an idle machine is faster.

In [ ]:
fits = {}
fits["linear_regression"] = engine.fit_sklearn(models["linear_regression"], X_tr, y_tr)
fits["random_forest"] = engine.fit_sklearn(models["random_forest"], X_tr, y_tr)
print({k: v["train_time_s"] for k, v in fits.items()}, "s")

In [ ]:
fits["xgboost"] = engine.fit_xgboost(models["xgboost"], X_tr, y_tr, X_val, y_val)
booster = fits["xgboost"]["booster"]              # trimmed to the best round; this is what gets exported
print(f"best round {fits['xgboost']['best_iteration'] + 1} of {fits['xgboost']['n_rounds']} "
      f"({fits['xgboost']['train_time_s']} s)")

In [ ]:
train_loader, val_loader = engine.make_loaders(scaler.transform(X_tr), y_tr, scaler.transform(X_val), y_val,
                                               cfg.batch_size, torch_device)
fits["mlp"] = engine.train_mlp(models["mlp"], train_loader, val_loader, cfg, torch_device)
print(f"{fits['mlp']['epochs_run']} epochs in {fits['mlp']['train_time_s']} s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (hist, a, b, title) in zip(axes, (
        (fits["xgboost"]["history"], "train_rmse", "val_rmse", "XGBoost RMSE per round"),
        (fits["mlp"]["history"], "train_loss", "val_loss", "MLP MSE per epoch"))):
    ax.plot(hist[a], label="train")
    ax.plot(hist[b], label="validation")
    ax.set_title(title)
    ax.legend()
plt.show()

## 6. Evaluation
Validation metrics choose the default model (lowest validation RMSE among the exported XGBoost and MLP); test
metrics are then reported for every experiment. RMSE and MAE are in $100k (the `_usd` columns in dollars).

In [ ]:
trained = {"linear_regression": models["linear_regression"], "random_forest": models["random_forest"],
           "xgboost": booster, "mlp": models["mlp"]}
val_metrics = engine.evaluate(trained, X_val, y_val, scaler, torch_device)
test_metrics = engine.evaluate(trained, X_test, y_test, scaler, torch_device)
best = engine.select_best(val_metrics, model_builder.EXPORTED)
print("default model:", best)
pd.concat({"validation": pd.DataFrame(val_metrics).T, "test": pd.DataFrame(test_metrics).T}, axis=1).round(4)

In [ ]:
best_pred = engine.predict(best, trained[best], X_test, scaler, torch_device)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, best_pred, s=4, alpha=0.3)
ax.plot([0, 5.2], [0, 5.2], "--", color="#eb6834", label="perfect prediction")
ax.set(xlabel="actual ($100k)", ylabel="predicted ($100k)", title=f"{best} on the test split")
ax.legend()
plt.show()

## 7. Inference
Exactly what the Space and the endpoint run: the models are saved with `export.save_models` into a scratch folder
(`outputs/.../staged_model`, same files as section 8), loaded back with `model.load(dir, device)`, and asked to
price the example districts from `../space/examples/districts.csv` (real test-split districts near five cities).
The last check confirms the reloaded files reproduce the test RMSE of section 6.

In [ ]:
config = export.build_config(cfg, best, val_metrics, fits["xgboost"], fits["mlp"])
staged = export.save_models(cfg.outputs_dir / "staged_model", booster, models["mlp"], scaler, config)
predictor = M.load(staged, device)

examples = pd.read_csv(utils.SPACE_DIR / "examples" / "districts.csv")
rows = []
for _, row in examples.iterrows():
    district = {k: float(row[k]) for k in M.RAW_FEATURES}
    rows.append({"city": row["city"], "actual_usd": row[M.TARGET] * M.TARGET_UNIT_USD,
                 **{f"{name}_usd": predictor.predict(district, model=name)["price_usd"] for name in predictor.choices[1:]}})
print(predictor.predict({k: float(examples.loc[0, k]) for k in M.RAW_FEATURES}))
display(pd.DataFrame(rows).round(0))

for name in predictor.choices[1:]:
    reloaded = engine.regression_metrics(y_test, predictor.predict_value(splits["test"], name))
    print(f"{name}: test RMSE in memory {test_metrics[name]['rmse']:.6f} | reloaded {reloaded['rmse']:.6f}")
    assert abs(reloaded["rmse"] - test_metrics[name]["rmse"]) < 1e-4

## 8. Export
Writes `xgb.ubj`, `model.safetensors` + `config.json`, `scaler.joblib`, `metrics.json` and the graphs in `assets/`
to `cfg.model_dir` (`../model` for a real run, `outputs/smoke/model` for a smoke run), then refreshes the
model card's metrics and comparison tables. To publish, set `PUSH_TO_HUB = True` in section 2; the token comes
from `hf auth login` (local) or an `HF_TOKEN` secret / environment variable. Smoke runs are never uploaded.

In [ ]:
metrics = export.build_metrics(
    best, val_metrics, test_metrics,
    data={"n_train": len(y_tr), "n_val": len(y_val), "n_test": len(y_test), "n_features": len(M.FEATURES)},
    fits=fits, train_time_s=sum(f["train_time_s"] for f in fits.values()), device=torch_device.type,
    smoke=cfg.smoke, source=f"retrained {utils.today()} with training/notebook.ipynb ({torch_device.type})",
    xgb_trees=booster.num_boosted_rounds(), mlp_params=utils.count_params(models["mlp"]),
)
export.export(cfg.model_dir, booster=booster, mlp=models["mlp"], scaler=scaler, config=config, metrics=metrics,
              xgb_history=fits["xgboost"]["history"], mlp_history=fits["mlp"]["history"],
              y_test=y_test, best_pred=best_pred)
for name in ("model_comparison", "training_curves", "pred_vs_actual", "feature_importance"):
    display(Image(filename=str(cfg.model_dir / "assets" / f"{name}.png")))

In [ ]:
for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts and ".git" not in path.parts:
        print(f"{path.relative_to(cfg.model_dir).as_posix():32s} {path.stat().st_size / 1e3:9.1f} KB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))